# AutoData Agent Colab Runner

This notebook runs the AutoData research pipeline from the plan:

`evaluate -> diagnose -> plan -> generate -> verify -> mix -> fine-tune -> re-evaluate`

Start with `RUN_MODE = "smoke"`. After that succeeds, switch to `prototype` with real MedMCQA and Qwen evaluation/training.

## 1. No Google Drive Required

This notebook saves outputs to the Colab runtime by default, so it does not mount or require Google Drive.

In [1]:
from pathlib import Path

LOCAL_OUTPUT_DIR = Path('/content/autodata_outputs')
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Outputs will be saved under:', LOCAL_OUTPUT_DIR)

Outputs will be saved under: /content/autodata_outputs


## 2. Locate Or Clone The Repository

If you opened this notebook directly in Colab, it clones the repository into `/content/autodata-agent`.

In [2]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/Jiaqi-Ye/AutoData.git'
REPO_DIR = '/content/autodata-agent'

current = Path.cwd()
if (current / 'autodata').exists() and (current / 'configs').exists():
    repo_path = current
else:
    repo_path = Path(REPO_DIR)
    if not (repo_path / 'autodata').exists():
        if not REPO_URL.strip():
            raise RuntimeError(
                'Set REPO_URL to your autodata-agent GitHub repo.'
            )
        if repo_path.exists():
            shutil.rmtree(repo_path)
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_path)], check=True)

os.chdir(repo_path)
print('Repository:', Path.cwd())
print('Files:', sorted(p.name for p in Path.cwd().iterdir())[:10])

Repository: /content/autodata-agent
Files: ['.git', '.gitignore', '.pytest_cache', 'AGENTS.md', 'LICENSE', 'README.md', 'autodata', 'configs', 'notebooks', 'outputs']


## 3. Install Dependencies

Smoke mode installs only lightweight packages. Prototype/full mode needs the ML stack. On Colab GPU, `bitsandbytes` is supported.

In [3]:
INSTALL_DEPS = True

LIGHTWEIGHT_DEPS = [
    'pyyaml>=6.0',
    'pytest>=7.0',
    'tqdm>=4.66',
]

REAL_DEPS = [
    'pyyaml>=6.0',
    'pytest>=7.0',
    'tqdm>=4.66',
    'datasets>=2.18',
    'torch>=2.1',
    'transformers>=4.43',
    'accelerate>=0.31',
    'peft>=0.11',
    'trl>=0.9',
    'bitsandbytes>=0.43',
]

OPENAI_DEPS = [
    'openai>=1.0',
]

if INSTALL_DEPS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *LIGHTWEIGHT_DEPS])
print('Dependency step complete.')

Dependency step complete.


## 4. Choose Run Settings

Recommended first run: keep the defaults and run smoke. For the real prototype, use:

- `RUN_MODE = 'prototype'`
- `USE_REAL_MODEL = True`
- `USE_REAL_TRAINING = True`
- keep `USE_MOCK_GENERATION = True` for a cheaper first real-model test, or set it to `False` with `GENERATION_PROVIDER = 'local_hf'` when you are ready to generate with Qwen2.5-7B.
- set `USE_MEDICAL_CRITIC = True` with `MEDICAL_CRITIC_PROVIDER = 'openai'` to review generated samples with your GPT API key before training.

In [4]:
RUN_MODE = "prototype"
RUN_STRATEGY_COMPARISON = False

USE_REAL_MODEL = True
USE_REAL_TRAINING = False
USE_MOCK_GENERATION = False
GENERATION_PROVIDER = "local_hf"

USE_MEDICAL_CRITIC = True
MEDICAL_CRITIC_PROVIDER = "openai"
MEDICAL_CRITIC_MODEL = "gpt-4o-mini"
MEDICAL_CRITIC_LOCAL_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MEDICAL_CRITIC_API_KEY_ENV = "OPENAI_API_KEY"
MEDICAL_CRITIC_FAIL_CLOSED = True

OUTPUT_DIR = "/content/autodata_outputs"

EVAL_SAMPLES_PER_DOMAIN = 20
TRAIN_POOL_SAMPLES_PER_DOMAIN = 100
TOTAL_SYNTHETIC_BUDGET = 25
MIN_SAMPLES_PER_DOMAIN = 3
MAX_TRAINING_STEPS = 10

print("Run mode:", RUN_MODE)
print("Strategy comparison:", RUN_STRATEGY_COMPARISON)
print("Medical critic:", USE_MEDICAL_CRITIC, MEDICAL_CRITIC_PROVIDER)

Run mode: prototype
Strategy comparison: False


## 5. Install Real-Run Dependencies If Needed

This cell only installs the larger ML stack when your settings require it.

In [5]:
needs_medical_critic_stack = USE_MEDICAL_CRITIC and MEDICAL_CRITIC_PROVIDER in {"local_hf", "qwen", "local_qwen"}
needs_openai_stack = USE_MEDICAL_CRITIC and MEDICAL_CRITIC_PROVIDER == "openai"
needs_real_stack = USE_REAL_MODEL or USE_REAL_TRAINING or (not USE_MOCK_GENERATION) or needs_medical_critic_stack
if INSTALL_DEPS and needs_real_stack:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REAL_DEPS])
if INSTALL_DEPS and needs_openai_stack:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *OPENAI_DEPS])
print('Real-stack needed:', needs_real_stack)
print('OpenAI client needed:', needs_openai_stack)

Real-stack needed: True


## 6. Build Runtime Config

This creates a temporary config from `configs/{RUN_MODE}_colab.yaml` and applies the notebook controls above.

In [6]:
import os
import yaml
from pathlib import Path

from autodata.config import save_config

config_path = Path("configs") / f"{RUN_MODE}_colab.yaml"
if not config_path.exists():
    raise FileNotFoundError(f"Missing config: {config_path}")

with config_path.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

config["project"]["output_dir"] = OUTPUT_DIR

config["models"]["use_real_model"] = bool(USE_REAL_MODEL)

# Use real MedMCQA when USE_REAL_MODEL=True.
config["dataset"]["use_mock_data"] = not bool(USE_REAL_MODEL)

# Generation settings.
config["generation"]["use_mock_generation"] = bool(USE_MOCK_GENERATION)
config["generation"]["provider"] = "mock" if USE_MOCK_GENERATION else GENERATION_PROVIDER

# For local_hf generation, start with the smaller Qwen model to avoid Colab OOM.
if not USE_MOCK_GENERATION and GENERATION_PROVIDER == "local_hf":
    config["models"]["generation_model"] = "Qwen/Qwen2.5-1.5B-Instruct"

# Optional LLM medical critic. Do not store API keys in the notebook or config.
if USE_MEDICAL_CRITIC and MEDICAL_CRITIC_PROVIDER == "openai" and not os.environ.get(MEDICAL_CRITIC_API_KEY_ENV):
    try:
        from google.colab import userdata

        api_key = userdata.get(MEDICAL_CRITIC_API_KEY_ENV)
        if api_key:
            os.environ[MEDICAL_CRITIC_API_KEY_ENV] = api_key
    except Exception:
        pass
if USE_MEDICAL_CRITIC and MEDICAL_CRITIC_PROVIDER == "openai" and not os.environ.get(MEDICAL_CRITIC_API_KEY_ENV):
    raise RuntimeError(
        f"Set {MEDICAL_CRITIC_API_KEY_ENV} in Colab Secrets or os.environ before using the OpenAI medical critic."
    )

config["medical_critic"] = {
    "enabled": bool(USE_MEDICAL_CRITIC),
    "provider": MEDICAL_CRITIC_PROVIDER,
    "model": MEDICAL_CRITIC_MODEL,
    "local_model": MEDICAL_CRITIC_LOCAL_MODEL,
    "api_key_env": MEDICAL_CRITIC_API_KEY_ENV,
    "fail_closed": bool(MEDICAL_CRITIC_FAIL_CLOSED),
    "max_new_tokens": 384,
}

# Training settings.
config["training"]["enabled"] = bool(USE_REAL_TRAINING)
config["training"]["dry_run"] = not bool(USE_REAL_TRAINING)

if EVAL_SAMPLES_PER_DOMAIN:
    config["dataset"]["eval_samples_per_domain"] = int(EVAL_SAMPLES_PER_DOMAIN)

if TRAIN_POOL_SAMPLES_PER_DOMAIN:
    config["dataset"]["train_pool_samples_per_domain"] = int(TRAIN_POOL_SAMPLES_PER_DOMAIN)

if TOTAL_SYNTHETIC_BUDGET:
    config["generation"]["total_budget"] = int(TOTAL_SYNTHETIC_BUDGET)

if MIN_SAMPLES_PER_DOMAIN:
    config["planning"]["min_samples_per_domain"] = int(MIN_SAMPLES_PER_DOMAIN)

if MAX_TRAINING_STEPS:
    config["training"]["max_steps"] = int(MAX_TRAINING_STEPS)

runtime_config_path = Path("/content/autodata_runtime_config.yaml")
save_config(config, runtime_config_path)

print("Runtime config:", runtime_config_path)
print(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))

Runtime config: /content/autodata_runtime_config.yaml
project:
  name: AutoData Agent
  run_mode: prototype
  seed: 42
  output_dir: /content/autodata_outputs
dataset:
  name: openlifescienceai/medmcqa
  use_mock_data: false
  target_domains:
  - Anatomy
  - Pharmacology
  - Pathology
  - Microbiology
  - Physiology
  eval_samples_per_domain: 20
  train_pool_samples_per_domain: 100
models:
  base_model: Qwen/Qwen2.5-1.5B-Instruct
  generation_model: Qwen/Qwen2.5-1.5B-Instruct
  use_real_model: true
  use_vllm: false
generation:
  provider: local_hf
  use_mock_generation: false
  total_budget: 25
  samples_per_domain: 200
  max_new_tokens: 512
planning:
  strategy: weakness_based
  min_samples_per_domain: 3
mixture:
  strategy: weakness_based
training:
  enabled: false
  dry_run: true
  method: qlora
  max_seq_length: 512
  per_device_train_batch_size: 1
  gradient_accumulation_steps: 8
  learning_rate: 0.0002
  max_steps: 10
  lora_r: 8
  lora_alpha: 16
  lora_dropout: 0.05
  target_mo

## 7. Optional Smoke Tests

This verifies the package before running the pipeline. It is quick in smoke mode.

In [7]:
RUN_TESTS = True

if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

## 8. Run AutoData

Single run executes the full loop once. Strategy comparison runs one loop for each listed strategy and writes a comparison report.

In [8]:
from pathlib import Path

if RUN_STRATEGY_COMPARISON:
    from autodata.experiments.strategy_comparison import run_strategy_comparison

    strategy_list = [item.strip() for item in STRATEGIES.split(',') if item.strip()]
    run_output = run_strategy_comparison(config, strategies=strategy_list)
    print('Comparison directory:', run_output['comparison_dir'])
else:
    from autodata.loop.run_loop import run_autodata_loop

    result = run_autodata_loop(config)
    run_output = {'run_dir': result.run_dir}
    print('Run directory:', result.run_dir)
    print('Base accuracy:', f'{result.evaluation_base.overall_accuracy:.3f}')
    print('After accuracy:', f'{result.evaluation_after.overall_accuracy:.3f}')
    print('Training status:', result.training_report.status)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Run directory: /content/autodata_outputs/runs/20260615_080844
Base accuracy: 0.390
After accuracy: 0.390
Training status: skipped


%cd /content/autodata-agent
!git fetch origin main
!git reset --hard origin/main
!git log -1 --oneline

## 9. Inspect Results

The important files are `round_summary.json`, `data_plan.json`, `verification_report.json`, `mixture_report.json`, and `next_round_recommendation.json`.

In [9]:
import json
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

if RUN_STRATEGY_COMPARISON:
    comparison_path = Path(run_output['comparison_dir']) / 'strategy_comparison.json'
    comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
    rows = comparison['strategies']
    if pd:
        display(pd.DataFrame(rows)[[
            'strategy',
            'base_overall_accuracy',
            'after_overall_accuracy',
            'overall_gain',
            'weakest_domain_improvement',
            'strong_domain_drop',
            'accepted_ratio',
            'training_status',
        ]])
    else:
        print(json.dumps(rows, indent=2))
else:
    run_dir = Path(run_output['run_dir'])
    summary = json.loads((run_dir / 'round_summary.json').read_text(encoding='utf-8'))
    data_plan = json.loads((run_dir / 'data_plan.json').read_text(encoding='utf-8'))
    verification = json.loads((run_dir / 'verification_report.json').read_text(encoding='utf-8'))
    mixture = json.loads((run_dir / 'mixture_report.json').read_text(encoding='utf-8'))
    next_round = json.loads((run_dir / 'next_round_recommendation.json').read_text(encoding='utf-8'))

    print('Run dir:', run_dir)
    print('Summary:')
    print(json.dumps({
        'base_overall_accuracy': summary['base_overall_accuracy'],
        'after_overall_accuracy': summary['after_overall_accuracy'],
        'overall_gain': summary['overall_gain'],
        'generated_count': summary['generated_count'],
        'accepted_count': summary['accepted_count'],
        'mixture_sample_count': summary['mixture_sample_count'],
        'training_status': summary['training_status'],
        'next_round_focus_domains': summary['next_round_focus_domains'],
    }, indent=2))

    if pd:
        plan_rows = []
        for domain, item in data_plan['plan'].items():
            plan_rows.append({
                'domain': domain,
                'num_samples': item['num_samples'],
                'data_type': item['data_type'],
                'reason': item['reason'],
                'accepted': verification['accepted_by_domain'].get(domain, 0),
                'mixture': mixture['domain_distribution'].get(domain, 0),
                'gain': next_round['per_domain_gain'].get(domain, 0.0),
                'efficiency': next_round['learning_efficiency_by_domain'].get(domain, 0.0),
            })
        display(pd.DataFrame(plan_rows))
    else:
        print('Data plan:', json.dumps(data_plan, indent=2))

Run dir: /content/autodata_outputs/runs/20260615_080844
Summary:
{
  "base_overall_accuracy": 0.39,
  "after_overall_accuracy": 0.39,
  "overall_gain": 0.0,
  "generated_count": 25,
  "accepted_count": 13,
  "mixture_sample_count": 13,
  "training_status": "skipped",
  "next_round_focus_domains": [
    "Microbiology",
    "Pharmacology",
    "Anatomy",
    "Pathology",
    "Physiology"
  ]
}


,domain,num_samples,data_type,reason,accepted,mixture,gain,efficiency
0,Anatomy,5,concept explanation + MCQ,Middle-performance domain receives balanced co...,4,4,0.0,0.0
1,Pharmacology,6,MCQ-style with mechanism explanation,Lower baseline accuracy (0.35) gives this doma...,4,4,0.0,0.0
2,Pathology,4,case-style MCQ with explanation,Middle-performance domain receives balanced co...,3,3,0.0,0.0
3,Microbiology,6,organism and treatment MCQ,Lower baseline accuracy (0.30) gives this doma...,1,1,0.0,0.0
4,Physiology,4,mechanism-focused review MCQ,Preservation allocation for a strong or risk-p...,1,1,0.0,0.0


## 10. Peek At Generated Samples

Use this to sanity-check the generated SFT format before training.

In [10]:
if not RUN_STRATEGY_COMPARISON:
    run_dir = Path(run_output['run_dir'])
    sample_path = run_dir / 'verified_samples.jsonl'
    print('Sample file:', sample_path)
    with sample_path.open('r', encoding='utf-8') as handle:
        for idx, line in enumerate(handle):
            print(json.dumps(json.loads(line), indent=2)[:1200])
            if idx >= 2:
                break
else:
    print('Open each run_dir from the comparison table to inspect its verified_samples.jsonl file.')

Sample file: /content/autodata_outputs/runs/20260615_080844/verified_samples.jsonl
{
  "domain": "Anatomy",
  "instruction": "Question: Which muscle group primarily controls the movement of the shoulder blade?\nA. Latissimus dorsi\nB. Pectoralis major\nC. Serratus anterior\nD. Triceps brachii",
  "response": "The correct answer is C. \nExplanation: The serratus anterior muscle group is responsible for elevating and rotating the scapulae.",
  "source": "local_hf",
  "generation_model": "Qwen/Qwen2.5-1.5B-Instruct",
  "round_id": "round_1",
  "metadata": {
    "raw_output": "```json\n{\n  \"domain\": \"Anatomy\",\n  \"instruction\": \"Which muscle group primarily controls the movement of the shoulder blade?\",\n  \"response\": \"The correct answer is C. \\nExplanation: The serratus anterior muscle group is responsible for elevating and rotating the scapulae.\",\n  \"options\": [\"A. Latissimus dorsi\", \"B. Pectoralis major\", \"C. Serratus anterior\", \"D. Triceps brachii\"]\n}\n```",
 

## 11. Zip The Latest Run

This is optional. It creates a zip archive beside the run directory for download or sharing.

In [11]:
from shutil import make_archive

if not RUN_STRATEGY_COMPARISON:
    run_dir = Path(run_output['run_dir'])
    archive_path = make_archive(str(run_dir), 'zip', root_dir=run_dir)
    print('Created:', archive_path)
else:
    comparison_dir = Path(run_output['comparison_dir'])
    archive_path = make_archive(str(comparison_dir), 'zip', root_dir=comparison_dir)
    print('Created:', archive_path)

Created: /content/autodata_outputs/runs/20260615_080844.zip


In [12]:
import json
from pathlib import Path

run_dir = Path("/content/autodata_outputs/runs/20260615_051356")

summary = json.loads((run_dir / "round_summary.json").read_text())
evaluation = json.loads((run_dir / "evaluation_base.json").read_text())
diagnosis = json.loads((run_dir / "diagnosis.json").read_text())

print("Overall accuracy:", summary["base_overall_accuracy"])
print("Weakest domain:", evaluation["weakest_domain"])
print("Strongest domain:", evaluation["strongest_domain"])

print("\nPer-domain accuracy:")
for domain, metrics in evaluation["per_domain"].items():
    print(domain, metrics["accuracy"], f"({metrics['correct']}/{metrics['num_samples']})")

print("\nDiagnosis:")
print(json.dumps(diagnosis, indent=2))

Overall accuracy: 0.39
Weakest domain: Microbiology
Strongest domain: Physiology

Per-domain accuracy:
Anatomy 0.4 (8/20)
Microbiology 0.3 (6/20)
Pathology 0.4 (8/20)
Pharmacology 0.35 (7/20)
Physiology 0.5 (10/20)

Diagnosis:
{
  "weak_domains": [
    "Microbiology",
    "Pharmacology"
  ],
  "stable_domains": [
    "Physiology"
  ],
  "risk_prone_domains": [
    "Physiology"
  ],
  "rationale_by_domain": {
    "Anatomy": "Anatomy is near the middle of the domain distribution (0.40 accuracy).",
    "Microbiology": "Microbiology is below the current target threshold (0.30 accuracy over 20 samples).",
    "Pathology": "Pathology is near the middle of the domain distribution (0.40 accuracy).",
    "Pharmacology": "Pharmacology is below the current target threshold (0.35 accuracy over 20 samples).",
    "Physiology": "Physiology is comparatively strong (0.50 accuracy) and should receive preservation data."
  },
  "summary": "Weak domains: Microbiology, Pharmacology. Stable domains: Physio